In [12]:
#Loading in dataset
import pandas as pd
import numpy as np
import seaborn as sns
import re
from sklearn.preprocessing import LabelEncoder
df = sns.load_dataset("titanic")
df.head(100)



,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0,3,male,NaN,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
96,0,1,male,71.0,0,0,34.6542,C,First,man,True,A,Cherbourg,no,True
97,1,1,male,23.0,0,1,63.3583,C,First,man,True,D,Cherbourg,yes,False
98,1,2,female,34.0,0,1,23.0000,S,Second,woman,False,NaN,Southampton,yes,False


In [15]:
#1. Function to get rid of duplicates and NAs from a data frame.

def clean_dataframe(df):
    # Remove nas
    df_cleaned = df.dropna()
    
    # Remove dupes
    df_cleaned = df_cleaned.drop_duplicates()
    
    return df_cleaned

df_cleaned = clean_dataframe(df)
print(f"Original shape: {df.shape}, Cleaned shape: {df_cleaned.shape}")
print(df_cleaned.head())

Original shape: (891, 15), Cleaned shape: (181, 15)
    survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
1          1       1  female  38.0      1      0  71.2833        C  First   
3          1       1  female  35.0      1      0  53.1000        S  First   
6          0       1    male  54.0      0      0  51.8625        S  First   
10         1       3  female   4.0      1      1  16.7000        S  Third   
11         1       1  female  58.0      0      0  26.5500        S  First   

      who  adult_male deck  embark_town alive  alone  
1   woman       False    C    Cherbourg   yes  False  
3   woman       False    C  Southampton   yes  False  
6     man        True    E  Southampton    no   True  
10  child       False    G  Southampton   yes  False  
11  woman       False    C  Southampton   yes   True  


In [16]:
#2 Function to go through all numerical columns and identify outliers then replace them and  produce a DF with the cleaned data and another with the outliers.

def clean_outliers_all_numeric(df):
    cleanedoutliers_df = df.copy()
 #create an empty dataframe to store outliers
    outliers_df = pd.DataFrame()  
# get numerical columns
    numeric_columns = cleanedoutliers_df.select_dtypes(include=np.number).columns
    
    # Loop through each numerical column and build IQR
    for col in numeric_columns:
        Q1 = cleanedoutliers_df[col].quantile(0.25)
        Q3 = cleanedoutliers_df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        # Find outliers in current column where less than lower bound or higher than upper bound
        col_outliers = cleanedoutliers_df[(cleanedoutliers_df[col] < lower) | (cleanedoutliers_df[col] > upper)].copy()
        
        # Add a column so you know which column triggered the outlier
        if not col_outliers.empty:
            col_outliers['outlier_column'] = col
            outliers_df = pd.concat([outliers_df, col_outliers], ignore_index=True)
        
        # Replace outliers in cleaned dataframe by capping
        cleanedoutliers_df[col] = cleanedoutliers_df[col].clip(lower, upper)
    
    return cleanedoutliers_df, outliers_df

cleaned_df, outliers_df = clean_outliers_all_numeric(df)

print("Cleaned data:")
print(cleaned_df.head())

print("\nOutliers found:")
print(outliers_df.head())

Cleaned data:
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0    1.0      0   7.2500        S  Third   
1         1       1  female  38.0    1.0      0  65.6344        C  First   
2         1       3  female  26.0    0.0      0   7.9250        S  Third   
3         1       1  female  35.0    1.0      0  53.1000        S  First   
4         0       3    male  35.0    0.0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  

Outliers found:
   survived  pclass   sex   age  sibsp  parch     fare embarked   class  who  \
0         0       2  male  66.0    0.0      0  10.5000        S  Second  man   
1         0       1  male  6